In [ ]:
!pip install nlpaug -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 26.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
import json, random, numpy as np
from sklearn.metrics import accuracy_score

with open("/content/drive/MyDrive/NLP/oos-eval/data/data_full.json") as f:
    data = json.load(f)

train_texts  = [item[0] for item in data["train"]]
train_labels = [item[1] for item in data["train"]]
test_texts   = [item[0] for item in data["test"]]
test_labels  = [item[1] for item in data["test"]]

all_labels = sorted(set(train_labels))
label2id   = {l: i for i, l in enumerate(all_labels)}
num_labels = len(all_labels)
train_ids  = [label2id[l] for l in train_labels]
test_ids   = [label2id[l] for l in test_labels]

with open("/content/drive/MyDrive/NLP/bert-clinc/noisy_test_set2.json") as f:
    noisy_data = json.load(f)

print(f"Train: {len(train_texts)} | Test: {len(test_texts)} | Intents: {num_labels}")


kb_aug = nac.KeyboardAug(aug_char_p=0.15)
sp_aug = naw.SpellingAug(aug_p=0.2)

def generate_noisy(text):
    choice = random.choices(["keyboard", "spelling"], weights=[0.7, 0.3])[0]
    try:
        if choice == "keyboard":
            return kb_aug.augment(text)[0]
        else:
            return sp_aug.augment(text)[0]
    except:
        return text

print("Generating noisy training pairs...")
noisy_train_texts = [generate_noisy(t) for t in train_texts]
print(f"Example: '{train_texts[0]}' -> '{noisy_train_texts[0]}'")


class ContrastiveIntentDataset(Dataset):
    def __init__(self, clean_texts, noisy_texts, labels, tokenizer, max_length=64):
        self.clean_texts = clean_texts
        self.noisy_texts = noisy_texts
        self.labels      = labels
        self.tokenizer   = tokenizer
        self.max_length  = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        clean_enc = self.tokenizer(
            self.clean_texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt"
        )
        noisy_enc = self.tokenizer(
            self.noisy_texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt"
        )
        return {
            "clean_input_ids":      clean_enc["input_ids"].squeeze(),
            "clean_attention_mask": clean_enc["attention_mask"].squeeze(),
            "noisy_input_ids":      noisy_enc["input_ids"].squeeze(),
            "noisy_attention_mask": noisy_enc["attention_mask"].squeeze(),
            "label":                torch.tensor(self.labels[idx])
        }

# ── Model ──────────────────────────────────────────────────
class ContrastiveBERT(nn.Module):
    def __init__(self, num_labels, model_name="bert-base-uncased", temperature=0.07):
        super().__init__()
        self.bert        = BertModel.from_pretrained(model_name)
        self.classifier  = nn.Linear(768, num_labels)
        self.temperature = temperature
        self.dropout     = nn.Dropout(0.1)

    def get_embedding(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return self.dropout(out.last_hidden_state[:, 0, :])

    def forward(self, clean_input_ids, clean_attention_mask,
                noisy_input_ids, noisy_attention_mask, labels=None):

        clean_embed = self.get_embedding(clean_input_ids, clean_attention_mask)
        noisy_embed = self.get_embedding(noisy_input_ids, noisy_attention_mask)
        logits      = self.classifier(clean_embed)

        if labels is not None:
            ce_loss    = nn.CrossEntropyLoss()(logits, labels)

            clean_norm = F.normalize(clean_embed, dim=-1)
            noisy_norm = F.normalize(noisy_embed, dim=-1)
            batch_size = clean_norm.size(0)
            embeddings = torch.cat([clean_norm, noisy_norm], dim=0)

            sim_matrix = torch.matmul(embeddings, embeddings.T) / self.temperature
            mask       = torch.eye(2 * batch_size, device=embeddings.device).bool()
            sim_matrix.masked_fill_(mask, float('-inf'))

            labels_contrast = torch.cat([
                torch.arange(batch_size, 2 * batch_size),
                torch.arange(0, batch_size)
            ]).to(embeddings.device)

            cl_loss    = nn.CrossEntropyLoss()(sim_matrix, labels_contrast)
            total_loss = ce_loss + 0.3 * cl_loss

            return total_loss, logits, ce_loss.item(), cl_loss.item()

        return logits

MODEL_PATH = "/content/drive/MyDrive/NLP/bert-clinc/final_model"
tokenizer  = BertTokenizer.from_pretrained(MODEL_PATH)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import BertForSequenceClassification
pretrained = BertForSequenceClassification.from_pretrained(MODEL_PATH)

contrastive_model = ContrastiveBERT(num_labels=num_labels)
contrastive_model.bert.load_state_dict(pretrained.bert.state_dict())
contrastive_model.classifier.weight.data = pretrained.classifier.weight.data
contrastive_model.classifier.bias.data   = pretrained.classifier.bias.data
contrastive_model = contrastive_model.to(device)
print(f"Model on {device}")


train_dataset = ContrastiveIntentDataset(train_texts, noisy_train_texts, train_ids, tokenizer)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)


optimizer = AdamW(contrastive_model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * 3
)


print("Starting contrastive fine-tuning...")
for epoch in range(3):
    contrastive_model.train()
    total_loss = total_ce = total_cl = 0

    for step, batch in enumerate(train_loader):
        clean_input_ids      = batch["clean_input_ids"].to(device)
        clean_attention_mask = batch["clean_attention_mask"].to(device)
        noisy_input_ids      = batch["noisy_input_ids"].to(device)
        noisy_attention_mask = batch["noisy_attention_mask"].to(device)
        labels               = batch["label"].to(device)

        optimizer.zero_grad()
        loss, logits, ce, cl = contrastive_model(
            clean_input_ids, clean_attention_mask,
            noisy_input_ids, noisy_attention_mask, labels
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(contrastive_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        total_ce   += ce
        total_cl   += cl

        if step % 100 == 0:
            print(f"Epoch {epoch+1} Step {step}/{len(train_loader)} "
                  f"Loss: {loss.item():.4f} CE: {ce:.4f} CL: {cl:.4f}")

    print(f"Epoch {epoch+1} done — Avg Loss: {total_loss/len(train_loader):.4f}")


def evaluate(texts, label_ids):
    contrastive_model.eval()
    enc    = tokenizer(texts, truncation=True, padding=True, max_length=64, return_tensors="pt")
    ids_t  = enc["input_ids"]
    mask_t = enc["attention_mask"]
    preds_all, labels_all = [], []
    with torch.no_grad():
        for i in range(0, len(texts), 64):
            embed  = contrastive_model.get_embedding(ids_t[i:i+64].to(device), mask_t[i:i+64].to(device))
            logits = contrastive_model.classifier(embed)
            preds  = torch.argmax(logits, dim=-1).cpu().numpy()
            preds_all.extend(preds)
            labels_all.extend(label_ids[i:i+64])
    return accuracy_score(labels_all, preds_all)

baseline = {
    "original": 0.9622, "casing": 0.9622, "keyboard": 0.4331,
    "spelling": 0.8284, "synonyms": 0.9622, "abbreviations": 0.9531,
    "whisper": 0.9236
}

noise_variants = {
    "original":    test_texts,
    "casing":      noisy_data["casing"],
    "keyboard":    noisy_data["keyboard"],
    "spelling":    noisy_data["spelling"],
    "synonyms":    noisy_data["synonyms"],
    "abbreviations": noisy_data["abbreviations"],
    "whisper":     noisy_data["whisper"],
}

print(f"\n{'Noise Type':<20} {'Baseline':>10} {'Contrastive':>13} {'Δ':>8}")
print("-" * 55)
for noise_type, texts in noise_variants.items():
    acc         = evaluate(texts, test_ids[:len(texts)])
    base        = baseline[noise_type]
    improvement = (acc - base) * 100
    print(f"{noise_type:<20} {base*100:>9.2f}% {acc*100:>12.2f}% {improvement:>+7.2f}%")

# ── Save ───────────────────────────────────────────────────
save_path = "/content/drive/MyDrive/NLP/bert-clinc/contrastive_model"
contrastive_model.bert.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
torch.save(contrastive_model.classifier.state_dict(), save_path + "/classifier.pt")
print(f"\nSaved to {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: 15000 | Test: 4500 | Intents: 150
Generating noisy training pairs...
Example: 'what expression would i use to say i love you if i were an italian' -> 'wha% exprrssi9n wLuld i use to say i love you if i Eere an Utaliah'


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model on cuda
Starting contrastive fine-tuning...
Epoch 1 Step 0/938 Loss: 5.2894 CE: 5.0675 CL: 0.7397
Epoch 1 Step 100/938 Loss: 5.1788 CE: 4.8375 CL: 1.1377
Epoch 1 Step 200/938 Loss: 4.6559 CE: 4.4381 CL: 0.7260
Epoch 1 Step 300/938 Loss: 3.8787 CE: 3.6382 CL: 0.8016
Epoch 1 Step 400/938 Loss: 2.9405 CE: 2.7447 CL: 0.6528
Epoch 1 Step 500/938 Loss: 2.1534 CE: 1.9138 CL: 0.7985
Epoch 1 Step 600/938 Loss: 0.8533 CE: 0.7638 CL: 0.2984
Epoch 1 Step 700/938 Loss: 0.7928 CE: 0.5185 CL: 0.9144
Epoch 1 Step 800/938 Loss: 0.5548 CE: 0.3834 CL: 0.5713
Epoch 1 Step 900/938 Loss: 0.3030 CE: 0.1588 CL: 0.4807
Epoch 1 done — Avg Loss: 2.6105
Epoch 2 Step 0/938 Loss: 0.2212 CE: 0.1572 CL: 0.2133
Epoch 2 Step 100/938 Loss: 0.2258 CE: 0.1029 CL: 0.4096
Epoch 2 Step 200/938 Loss: 0.2551 CE: 0.1029 CL: 0.5074
Epoch 2 Step 300/938 Loss: 0.0665 CE: 0.0630 CL: 0.0118
Epoch 2 Step 400/938 Loss: 0.0777 CE: 0.0605 CL: 0.0576
Epoch 2 Step 500/938 Loss: 0.0918 CE: 0.0394 CL: 0.1747
Epoch 2 Step 600/938 Loss:

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved to /content/drive/MyDrive/NLP/bert-clinc/contrastive_model
